# Аксиора: полный запуск OCR одной командой

Выберите аккаунт с нужным Drive, **T4 GPU** и нажмите **Среда выполнения → Выполнить все**.
Когда Colab запросит доступ к Drive, подтвердите его. Для автоматической выгрузки на Hub
добавьте `HF_TOKEN` в Секреты Colab и разрешите ноутбуку доступ.
Ноутбук сам подготовит данные при отсутствии ZIP, обучит модель и сохранит контрольные точки.
При доступном `HF_TOKEN` веса автоматически отправятся на Hub; без ключа обучение завершится с экспортом на Drive.
Готовый ZIP используется повторно, а совместимое обучение продолжается с последней контрольной точки.

Чат ChatGPT после старта можно закрыть: вычисления выполняются в Colab. Саму вкладку Colab лучше оставить
открытой. Непрерывное выполнение после её закрытия зависит от тарифа и условий Colab; Google AI Pro
не следует считать гарантией фонового выполнения. При обрыве можно восстановиться с Drive.

Основной профиль: полный MathWriting, 12 000 train, 1000 validation, 500 test, одна T4, FP16, три эпохи.
Подготовка полного архива требует загрузки около 2,9 ГБ. MathWriting имеет некоммерческую лицензию CC BY-NC-SA 4.0.
Для быстрой проверки выберите `RUN_MODE = "smoke"` и `DATASET_VARIANT = "excerpt"`; эти веса не пригодны для учеников.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/danilsamchenckodmitrievich-netizen/Laba-proektx.git"
SOURCE_REF = "main"  # Можно указать ветку; фактический commit попадёт в отчёт.
REPO_DIR = Path("/content/axiora")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", SOURCE_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    actual_origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if actual_origin != REPO_URL:
        raise RuntimeError("В /content/axiora находится другой проект. Выберите новый REPO_DIR.")
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "training/requirements-colab.txt")], check=True)
print("Версия исходников:", SOURCE_COMMIT)


In [ ]:
USE_GOOGLE_DRIVE = True
RUN_MODE = "t4"  # "smoke" для excerpt и двух проверочных шагов.
RUN_NAME = "trocr-small-v2"  # Новые данные/гиперпараметры требуют нового имени.
RESUME = True
HUB_REPO_ID = None  # None → <ваш-логин>/axiora-ocr, для smoke добавляется -smoke.
HUB_PRIVATE = True
PUSH_TO_HUB = True
BENCHMARK_INT8 = True
DATASET_DIR = Path("/content/axiora-ocr-data")


DATASET_VARIANT = "full"
PREPARE_LIMITS = {"train": 12000, "validation": 1000, "test": 500}
SYNTHETIC_COUNT = 300
DATASET_DIR = Path("/content") / ("axiora-ocr-data-" + DATASET_VARIANT)


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive/Axiora/ocr")
else:
    PERSIST_DIR = Path("/content/axiora-artifacts")
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print("Артефакты и контрольные точки:", PERSIST_DIR)


In [ ]:
import torch
from packaging.version import Version
from training.ocr import TrainConfig, environment_report
from transformers import set_seed

if Version(torch.__version__.split("+")[0]) < Version("2.6.0"):
    raise RuntimeError("Нужен PyTorch >= 2.6 для безопасной загрузки исходных весов. Выберите актуальную среду Colab.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU не подключён. Выберите T4 GPU в настройках среды выполнения.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Этот ноутбук настроен на одну GPU. Выберите среду с одной T4.")
config = TrainConfig(mode=RUN_MODE, **({"train_batch_size": 1, "gradient_accumulation_steps": 1, "eval_samples": 2} if RUN_MODE == "smoke" else {}))
set_seed(config.seed)
print(environment_report())
print("VRAM, ГБ:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("Эффективный batch:", config.train_batch_size * config.gradient_accumulation_steps)


## Проверка доступа к Hub до долгого обучения

Проверяем наличие `HF_TOKEN` и авторизацию заранее. Сам ключ не печатается и не сохраняется.
Для выгрузки создайте токен с правом записи. При отсутствии ключа или ошибке авторизации обучение
продолжается с экспортом на Drive, а выгрузка откладывается до настройки доступа.


In [ ]:
if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import HfApi
    def check_hub_access():
        try:
            token = userdata.get("HF_TOKEN")
        except Exception:
            return None
        if not token:
            return None
        try:
            return HfApi(token=token).whoami()["name"]
        except Exception:
            return None
    hub_owner = check_hub_access()
    if hub_owner:
        print("Аккаунт Hub:", hub_owner)
    else:
        PUSH_TO_HUB = False
        print("Доступ к Hub пока не подтверждён. Обучение продолжится; веса сохранятся на Drive.")
        print("Для последующей выгрузки добавьте HF_TOKEN, включите PUSH_TO_HUB и выполните последнюю ячейку.")
else:
    print("Выгрузка отключена: готовые веса сохранятся на Drive.")


## Данные: восстановление или автоматическая подготовка

Если ZIP на Drive уже есть, восстанавливаем его. Иначе загружаем MathWriting, формируем выборки
и сохраняем ZIP для последующих сессий. Отдельно открывать первый ноутбук не требуется.


In [ ]:
from training.data import (download_mathwriting, prepare_dataset, zip_dataset,
                           unpack_dataset, audit_dataset, sha256_file)
import time

DATASET_ZIP = PERSIST_DIR / ("axiora-ocr-dataset-" + DATASET_VARIANT + ".zip")
original_zip = PERSIST_DIR / "axiora-ocr-dataset.zip"
if not DATASET_ZIP.exists() and original_zip.exists():
    import zipfile
    import json
    with zipfile.ZipFile(original_zip) as existing_archive:
        original_variant = json.loads(existing_archive.read("dataset_report.json"))["variant"]
    if original_variant == DATASET_VARIANT or original_variant == "custom":
        DATASET_ZIP = original_zip
if not DATASET_DIR.exists():
    if DATASET_ZIP.exists():
        checksum_file = DATASET_ZIP.with_suffix(".zip.sha256")
        if checksum_file.exists() and sha256_file(DATASET_ZIP) != checksum_file.read_text().strip():
            raise RuntimeError("SHA-256 архива не совпадает. Восстановите целый архив перед продолжением.")
        unpack_dataset(DATASET_ZIP, DATASET_DIR)
    else:
        print("Готового набора нет. Автоматически подготавливаем MathWriting:", DATASET_VARIANT)
        last_message = [0.0]
        def progress(written, total):
            if time.monotonic() - last_message[0] > 5:
                print(f"Загружено {written / 2**20:.0f} МБ" + (f" из {total / 2**20:.0f}" if total else ""))
                last_message[0] = time.monotonic()
        archive_path = download_mathwriting(DATASET_VARIANT, Path("/content") / f"mathwriting-{DATASET_VARIANT}.tgz", progress)
        prepare_dataset(archive_path, DATASET_DIR, PREPARE_LIMITS,
                        variant=DATASET_VARIANT, synthetic_count=SYNTHETIC_COUNT, seed=config.seed)
dataset_audit = audit_dataset(DATASET_DIR)
if RUN_MODE == "t4" and dataset_audit["report"]["variant"] == "excerpt":
    raise RuntimeError("Найден excerpt: выберите smoke либо подготовьте full в новом DATASET_DIR и DATASET_ZIP.")
if not DATASET_ZIP.exists():
    checksum = zip_dataset(DATASET_DIR, DATASET_ZIP)
    DATASET_ZIP.with_suffix(".zip.sha256").write_text(checksum + "\n")
print("Выборки:", dataset_audit["counts"])
print("Отпечаток данных:", dataset_audit["fingerprint"])


## Модель и аудит разметки

Используется закреплённая версия `microsoft/trocr-small-handwritten` (около 61 млн параметров).
Исходная модель читает обычный почерк; математические формулы она изучит при дообучении.
Словарь поддерживает стандартные LaTeX-команды. Неподдерживаемые или слишком длинные подписи явно
исключаются и учитываются в отчёте: тихого обрезания правильного ответа нет.


In [ ]:
from dataclasses import asdict
from training.ocr import load_base_model, filter_labels, FormulaDataset, make_trainer, ensure_run_context, training_code_fingerprint
from training.data import load_manifest
import random

model, processor = load_base_model(config)
records, label_audit = {}, {}
for split in ("train", "validation", "test"):
    records[split], label_audit[split] = filter_labels(load_manifest(DATASET_DIR, split), processor.tokenizer, config.max_target_length)
print("Исключения по длине/словарю:", label_audit)
print("Осталось:", {key: len(value) for key, value in records.items()})
print("Параметров:", sum(parameter.numel() for parameter in model.parameters()))
validation_rows = random.Random(config.seed).sample(records["validation"], min(config.eval_samples, len(records["validation"])))
datasets = {
    "train": FormulaDataset(DATASET_DIR, records["train"], processor, augment=True),
    "validation": FormulaDataset(DATASET_DIR, validation_rows, processor),
}
RUN_DIR = PERSIST_DIR / "runs" / RUN_MODE / RUN_NAME
context = {"dataset_fingerprint": dataset_audit["fingerprint"], "training_code_fingerprint": training_code_fingerprint(), "config": asdict(config)}
ensure_run_context(RUN_DIR, context, RESUME)
trainer = make_trainer(model, processor, datasets, config, RUN_DIR, fp16=True)


## Обучение и возобновление

Контрольные точки содержат модель, optimizer, scheduler и состояние случайных чисел. Сохраняются до двух
точек, включая лучшую. При обрыве запустите ноутбук заново с теми же данными, настройками и `RUN_NAME`.
Выбор лучшей точки использует только validation и CER. Test пока не используется для выбора модели.

Если возникнет CUDA out of memory, задайте в ячейке создания `TrainConfig` batch 1 и накопление 16,
выберите новый `RUN_NAME` и повторите запуск. Не меняйте настройки внутри существующего run.


In [ ]:
from transformers.trainer_utils import get_last_checkpoint
checkpoint = get_last_checkpoint(str(RUN_DIR)) if RESUME else None
print("Продолжение:", checkpoint or "новое обучение")
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=checkpoint)
trainer.save_state()
peak_vram_gib = torch.cuda.max_memory_allocated() / 2**30
print("Пиковое выделение VRAM, ГБ:", round(peak_vram_gib, 2))


## Финальная оценка и сохранение

После выбора контрольной точки оцениваем отложенный test. CER измеряет ошибки символов, exact match —
совпадение записи; эти метрики не означают правильность математического решения. Отдельная метрика
игнорирует только пробелы. Отчёт содержит число исключённых длинных/неподдерживаемых формул.


In [ ]:
from training.ocr import export_model
from training.data import json_write
test_dataset = FormulaDataset(DATASET_DIR, records["test"], processor)
# Inspect held-out validation examples alongside aggregate scores.
sample_rows = validation_rows[:12]
preview = trainer.predict(FormulaDataset(DATASET_DIR, sample_rows, processor))
import numpy as np
token_ids = np.where(preview.predictions == -100, processor.tokenizer.pad_token_id, preview.predictions)
texts = processor.batch_decode(token_ids, skip_special_tokens=True)
validation_examples = [{"id": row["id"], "expected": row["latex"], "predicted": text}
                       for row, text in zip(sample_rows, texts)]
for example in validation_examples:
    print(example["expected"], "→", example["predicted"])
test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
run_report = {
    "schema_version": 1, "config": asdict(config), "source_commit": SOURCE_COMMIT,
    "training_code_fingerprint": training_code_fingerprint(),
    "dataset": dataset_audit, "label_audit": label_audit,
    "retained_counts": {split: len(rows) for split, rows in records.items()},
    "validation_selection_count": len(validation_rows), "validation_examples": validation_examples,
    "train_metrics": train_result.metrics, "test_metrics": test_metrics,
    "best_checkpoint": Path(trainer.state.best_model_checkpoint).name if trainer.state.best_model_checkpoint else None,
    "environment": environment_report(), "peak_vram_gib": peak_vram_gib,
    "intended_input": "one cropped handwritten formula; not a full page or proof",
}
EXPORT_DIR = export_model(trainer.model, processor, RUN_DIR / "export", run_report)
print("Сохранено:", EXPORT_DIR)
print(test_metrics)


## CPU: скорость и необязательный INT8

Отдельный процесс загружает экспорт на CPU, ограничивает PyTorch двумя потоками и измеряет 12 формул
из validation после прогрева. Также сравниваются FP32 и динамический INT8, включая CER и совпадение предсказаний.
Медиана и p95 не включают сеть и холодный запуск Space. Даже результат быстрее двух секунд нужно
перепроверить на настоящем CPU Basic: два потока в Colab не равны аппаратной квоте Space.
INT8 не выбирается автоматически: при ухудшении CER или совпадения предсказаний отчёт помечает регрессию качества.
Если необязательное измерение не завершилось, веса сохранятся, а ошибка будет явно записана в отчёт.


In [ ]:
import os
command = [sys.executable, "-m", "training.benchmark", "--model-dir", str(EXPORT_DIR),
           "--dataset-dir", str(DATASET_DIR), "--output", str(EXPORT_DIR / "cpu_benchmark.json"),
           "--samples", "12", "--max-length", str(config.max_target_length)]
if BENCHMARK_INT8:
    command.append("--int8")
cpu_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}
benchmark = subprocess.run(command, cwd=REPO_DIR, env=cpu_env, text=True, capture_output=True)
if benchmark.returncode:
    error = benchmark.stderr[-3000:]
    json_write(EXPORT_DIR / "cpu_benchmark_error.json", {"status": "failed", "error": error})
    print("CPU-проверка не завершилась; перенос на Space пока не подтверждён. Веса сохранены.")
    print(error)
else:
    print(benchmark.stdout)


## Автоматическая выгрузка на Hugging Face Hub

Последняя ячейка загружает веса через `push_to_hub`, затем согласованный процессор, карточку и отчёты.
По умолчанию репозиторий приватный. Для чтения приватных весов будущему Space потребуется `HF_TOKEN`
в его Secrets. Если авторизация или сеть прервались, поправьте доступ и повторите эту ячейку: заново обучать не нужно.
Не запускайте `smoke` с именем настоящей модели: для проверочных весов требуется суффикс `-smoke`.


In [ ]:
from training.ocr import publish_to_hub
import json
upload_dir = PERSIST_DIR / "runs" / RUN_MODE / RUN_NAME / "export"

if PUSH_TO_HUB:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    from google.colab import userdata
    if not (upload_dir / "training_report.json").exists():
        raise FileNotFoundError("Готовый экспорт не найден. Сначала завершите обучение и сохранение модели.")
    upload_report = json.loads((upload_dir / "training_report.json").read_text())
    upload_processor = TrOCRProcessor.from_pretrained(upload_dir, local_files_only=True, use_fast=False)
    upload_model = VisionEncoderDecoderModel.from_pretrained(upload_dir, local_files_only=True, use_safetensors=True)
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        raise RuntimeError("Добавьте HF_TOKEN в Секреты Colab и разрешите доступ этому ноутбуку. Веса уже сохранены на Drive.") from None
    try:
        uploaded_model = publish_to_hub(upload_model, upload_processor, upload_dir, upload_report,
                                        token=hf_token, repo_id=HUB_REPO_ID, private=HUB_PRIVATE)
    finally:
        del hf_token
    from training.data import json_write
    json_write(upload_dir.parent / "hub_upload.json", uploaded_model)
    print("Модель:", uploaded_model["url"])
    print("Версия для будущего Space:", uploaded_model["revision"])
    del upload_model, upload_processor
else:
    print("Выгрузка не выполнялась; каталог экспорта:", upload_dir)


Подготовка Этапа 2 завершена. Реальные веса и оценка качества появятся после вашего полного запуска в Colab.
Следующий этап разработки — FastAPI,
который загрузит указанную версию с Hub, применит тот же preprocessing и объединит OCR с SymPy и LLM-проверкой.
